# 02 · Regresión logística binaria programada desde cero

**Semana 2 de la planificación.** Antes de pasar a los 8 géneros, comprobamos que nuestra implementación funciona en casos sencillos:

1. **Ejemplo de clase** (5 clientes de un banco): ejecutamos nuestro código sobre los mismos datos de las diapositivas.
2. **Rock frente a Hip-Hop** con las 518 características: primer problema real, binario.
3. **Comparación con scikit-learn** (objetivo O1 de la propuesta): si nuestra implementación es correcta, debe dar los mismos coeficientes.
4. **Interpretación** de los coeficientes mediante razones de momios.

El código de la regresión logística está en `src/regresion_logistica.py`. Recordemos la regla de actualización de Newton-Raphson que hemos programado:

$$\beta_{t+1} = \beta_t + (X^T W_t X)^{-1} X^T (y - p_t)$$

y su versión con regularización ridge (penalización $\lambda\|\beta\|^2$), que usaremos más adelante:

$$\beta_{t+1} = \beta_t + (X^T W_t X + 2\lambda I)^{-1} \left[X^T (y - p_t) - 2\lambda\beta_t\right]$$

In [ ]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression

from src.datos import cargar_procesados
from src.regresion_logistica import RegresionLogistica, sigmoide, exactitud

CARPETA_FIGURAS = "../figuras"


## 1. Prueba con el ejemplo de clase

Usamos los datos de las diapositivas: ingreso mensual (en miles de euros) y si el cliente pagó el préstamo. Ejecutamos 15 iteraciones sin regularización ($\lambda = 0$) y mostramos una tabla como la de la diapositiva "Resumen de la convergencia".

In [ ]:
x_banco = np.array([1.5, 2.8, 4.2, 5.6, 7.3])
y_banco = np.array([0, 0, 1, 1, 1])

modelo_banco = RegresionLogistica(lam=0, max_iter=15).ajustar(x_banco.reshape(-1, 1), y_banco)

tabla = pd.DataFrame({
    "iteración": range(1, modelo_banco.n_iter + 1),
    "β0": [b[0] for b in modelo_banco.historial_beta],
    "β1": [b[1] for b in modelo_banco.historial_beta],
    "‖Δβ‖": modelo_banco.historial_paso,
    "log-verosimilitud": modelo_banco.historial_logver,
})
print("¿Ha convergido?", modelo_banco.convergio)
tabla.round(4)

**¿Qué está pasando?** El tamaño del paso ‖Δβ‖ no baja a cero: se queda estable alrededor de 5, y los coeficientes crecen sin parar. Mientras tanto, la log-verosimilitud se acerca cada vez más a 0, que es su valor máximo posible.

La explicación está en los datos: **los clientes que no pagan tienen ingresos de 2,8 o menos, y los que pagan, de 4,2 o más**. Cualquier frontera entre 2,8 y 4,2 separa perfectamente las dos clases. Cuanto más empinada es la sigmoide, más se acercan las probabilidades a 0 y 1 exactas y mayor es la verosimilitud, así que el máximo "está en el infinito": **no existe un estimador de máxima verosimilitud finito**. Es exactamente el caso de *separación completa* que avisa la diapositiva de "Errores comunes".

*Nota:* nuestra primera iteración da β ≈ (−3,108; 0,820). Al repetir el cálculo a mano obtenemos Σxᵢ² = 2,25 + 7,84 + 17,64 + 31,36 + 53,29 = 112,38, y con ese valor sale el mismo resultado que con el código.

La solución que propone la propia diapositiva de "Extensiones" es la **regularización ridge**. La probamos con un λ pequeño:

In [ ]:
modelo_ridge = RegresionLogistica(lam=0.1, max_iter=50).ajustar(x_banco.reshape(-1, 1), y_banco)

print("¿Ha convergido?", modelo_ridge.convergio, "en", modelo_ridge.n_iter, "iteraciones")
print("β =", modelo_ridge.beta.round(3))
print("Frontera de decisión: x =", round(-modelo_ridge.beta[0] / modelo_ridge.beta[1], 3), "miles de euros")

In [ ]:
x_curva = np.linspace(0, 9, 300)
plt.figure(figsize=(8, 4.5))
plt.scatter(x_banco, y_banco, color="black", zorder=3, label="clientes")
plt.plot(x_curva, sigmoide(modelo_banco.beta[0] + modelo_banco.beta[1] * x_curva),
         label=f"λ = 0, tras {modelo_banco.n_iter} iteraciones (sigue creciendo)")
plt.plot(x_curva, sigmoide(modelo_ridge.beta[0] + modelo_ridge.beta[1] * x_curva),
         label="λ = 0.1 (converge)")
plt.xlabel("Ingreso mensual (miles de euros)")
plt.ylabel("P(pagar)")
plt.title("Ejemplo de clase: sin regularización la sigmoide se vuelve un escalón")
plt.legend()
plt.tight_layout()
plt.savefig(f"{CARPETA_FIGURAS}/02_ejemplo_clase.png", dpi=150)
plt.show()

Sin regularización, la curva se convierte casi en un escalón: el modelo está "demasiado seguro". Con ridge, el término $-\lambda\|\beta\|^2$ castiga los coeficientes grandes, el máximo pasa a ser finito y el algoritmo converge.

## 2. Rock frente a Hip-Hop con las 518 características

Cargamos los datos preparados en el notebook 01 y nos quedamos con las canciones de esos dos géneros. Codificamos **Rock = 1** e **Hip-Hop = 0**.

In [ ]:
datos = cargar_procesados("../datos/procesados.npz")
nombres = datos["nombres"]

def filtrar_binario(X, y, positivo="Rock", negativo="Hip-Hop"):
    mascara = (y == positivo) | (y == negativo)
    return X[mascara], (y[mascara] == positivo).astype(float)

Xb_train, yb_train = filtrar_binario(datos["X_train"], datos["y_train"])
Xb_val, yb_val = filtrar_binario(datos["X_val"], datos["y_val"])
print("Entrenamiento:", Xb_train.shape, " Validación:", Xb_val.shape)

### 2.1 Sin regularización (λ = 0)

Además de la exactitud, calculamos el **número de condición** de la matriz XᵀWX de la última iteración. Mide cuánto se amplifican los errores de redondeo al resolver el sistema: un valor cercano a 1 es ideal, y valores enormes (10¹⁰ o más) indican que la matriz es casi singular.

In [ ]:
def resumen(modelo, nombre):
    print(f"--- {nombre} ---")
    print("  ¿Ha convergido?:", modelo.convergio, f"({modelo.n_iter} iteraciones)")
    if modelo.error:
        print("  Error:", modelo.error)
    print(f"  Tamaño de los coeficientes ‖β‖: {np.linalg.norm(modelo.beta):.2f}")
    print(f"  Número de condición de XᵀWX: {np.linalg.cond(modelo.matriz_newton):.2e}")
    print(f"  Exactitud entrenamiento: {exactitud(yb_train, modelo.predecir(Xb_train)):.3f}")
    print(f"  Exactitud validación:    {exactitud(yb_val, modelo.predecir(Xb_val)):.3f}")

modelo_sin = RegresionLogistica(lam=0, max_iter=30).ajustar(Xb_train, yb_train)
resumen(modelo_sin, "Sin regularización (λ = 0)")

**Cómo interpretar el resultado:** con 518 características y solo 1.600 canciones, es muy probable que Rock y Hip-Hop sean separables en el entrenamiento, como en el ejemplo del banco. Si es así, veréis que el modelo no converge, que ‖β‖ es muy grande, que la exactitud en entrenamiento es casi del 100 % y que la de validación es claramente menor. Eso es **sobreajuste**: el modelo memoriza el entrenamiento en lugar de aprender algo que generalice.

### 2.2 Con regularización (λ = 1)

In [ ]:
LAMBDA = 1.0
modelo_con = RegresionLogistica(lam=LAMBDA, max_iter=50).ajustar(Xb_train, yb_train)
resumen(modelo_con, f"Con ridge (λ = {LAMBDA})")

### 2.3 Convergencia de Newton-Raphson

Dibujamos ‖Δβ‖ en cada iteración con el eje vertical en **escala logarítmica**. Cerca del óptimo, Newton-Raphson converge de forma **cuadrática**: el número de cifras correctas aproximadamente se duplica en cada paso, así que la curva cae cada vez más deprisa.

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(13, 4.5))

ejes[0].semilogy(range(1, modelo_sin.n_iter + 1), modelo_sin.historial_paso, "o-", label="λ = 0")
ejes[0].semilogy(range(1, modelo_con.n_iter + 1), modelo_con.historial_paso, "o-", label=f"λ = {LAMBDA}")
ejes[0].set_xlabel("Iteración")
ejes[0].set_ylabel("‖Δβ‖ (escala logarítmica)")
ejes[0].set_title("Tamaño del paso de Newton")
ejes[0].legend()

ejes[1].plot(range(1, modelo_sin.n_iter + 1), modelo_sin.historial_logver, "o-", label="λ = 0")
ejes[1].plot(range(1, modelo_con.n_iter + 1), modelo_con.historial_logver, "o-", label=f"λ = {LAMBDA}")
ejes[1].set_xlabel("Iteración")
ejes[1].set_ylabel("Log-verosimilitud ℓ(β)")
ejes[1].set_title("Evolución de la log-verosimilitud")
ejes[1].legend()

fig.tight_layout()
fig.savefig(f"{CARPETA_FIGURAS}/02_convergencia_newton.png", dpi=150)
plt.show()

## 3. Comparación con scikit-learn (objetivo O1)

scikit-learn también tiene regresión logística con penalización ridge, pero usa otra forma de escribir el parámetro: en lugar de λ usa **C**, que es "al revés" (C grande = poca regularización). La equivalencia con nuestra fórmula es

$$C = \frac{1}{2\lambda}$$

Usamos su método `newton-cholesky`, que es también un método de Newton, con una tolerancia muy pequeña para que ambos lleguen al óptimo exacto.

In [ ]:
sk = LogisticRegression(C=1 / (2 * LAMBDA), solver="newton-cholesky", tol=1e-12, max_iter=1000)
sk.fit(Xb_train, yb_train)

beta_sklearn = np.concatenate([sk.intercept_, sk.coef_.ravel()])
diferencia = np.max(np.abs(modelo_con.beta - beta_sklearn))

print(f"Mayor diferencia entre coeficientes: {diferencia:.2e}")
print("¿Cumple O1 (diferencia < 1e-4)?", diferencia < 1e-4)
print("Predicciones iguales en validación:", np.mean(modelo_con.predecir(Xb_val) == sk.predict(Xb_val)) * 100, "%")

Si la diferencia es del orden de 10⁻⁶ o menor, nuestra implementación calcula lo mismo que una librería profesional. A partir de aquí podemos fiarnos de ella para el resto del proyecto.

## 4. ¿Qué hace que una canción suene a rock? Razones de momios

Como las características están estandarizadas, cada coeficiente βⱼ indica cuánto cambian los log-odds de "ser Rock" cuando la característica j **aumenta una desviación típica**. La **razón de momios** $e^{\beta_j}$ dice por cuánto se **multiplican** las odds:

- $e^{\beta_j} > 1$: la característica empuja hacia **Rock**.
- $e^{\beta_j} < 1$: empuja hacia **Hip-Hop**.

In [ ]:
coeficientes = pd.Series(modelo_con.beta[1:], index=nombres)   # quitamos el intercepto β0
mas_rock = coeficientes.sort_values(ascending=False).head(8)
mas_hiphop = coeficientes.sort_values().head(8)

tabla_or = pd.DataFrame({
    "β": pd.concat([mas_rock, mas_hiphop]),
    "razón de momios e^β": np.exp(pd.concat([mas_rock, mas_hiphop])),
})
tabla_or.round(3)

In [ ]:
seleccion = pd.concat([mas_hiphop, mas_rock[::-1]])
colores = ["tab:orange" if v < 0 else "tab:blue" for v in seleccion]

plt.figure(figsize=(9, 6))
plt.barh(seleccion.index, seleccion.values, color=colores)
plt.axvline(0, color="black", linewidth=0.8)
plt.xlabel("Coeficiente β (azul: hacia Rock, naranja: hacia Hip-Hop)")
plt.title("Características con más peso en Rock frente a Hip-Hop")
plt.tight_layout()
plt.savefig(f"{CARPETA_FIGURAS}/02_coeficientes_rock_hiphop.png", dpi=150)
plt.show()

**Precaución al interpretar:** como hay muchas características correlacionadas, el modelo puede repartir el efecto entre varias de ellas (por ejemplo, entre la media y la mediana de un mismo descriptor). Por eso conviene fijarse en qué *tipo* de descriptor aparece (MFCC, croma, centroide espectral…) más que en una columna concreta.

## Resumen

- Nuestra implementación reproduce el ejemplo de clase y muestra por qué ese ejemplo no converge: los datos son separables.
- Con 518 características, Rock y Hip-Hop se comportan igual, y la regularización ridge es necesaria para que el método converja y no sobreajuste.
- Nuestros coeficientes coinciden con los de scikit-learn, así que la implementación es correcta (objetivo O1).
- Las razones de momios nos dan una primera respuesta a la pregunta del título.